# ASCII Aventura — Gerar APK no celular

Abra no Google Colab e pressione **▶** na célula abaixo. Conceda acesso ao ambiente de execução quando solicitado. O APK será compilado e baixado para o celular. Não é preciso instalar Android Studio nem usar computador. O notebook usa apenas código do repositório público e ferramentas oficiais Android.

**Observação:** o processo requer uma conta Google e conexão com a internet. O APK resultante será do tipo *debug*, adequado para instalação e testes, não para publicação na Play Store.


In [ ]:
# Compilacao do ASCII Aventura pelo Google Colab (Android ou computador)
# Execute esta celula com a conta Google. Ao final, o APK sera baixado automaticamente.
import os, sys, shutil, zipfile, subprocess, urllib.request
from pathlib import Path
from google.colab import files

def run(args, **kwargs):
    print(">>", " ".join(map(str,args)), flush=True)
    return subprocess.run(args, check=True, **kwargs)

root=Path("/content/ascii-aventura")
sdk=Path("/content/android-sdk")
gradle_dir=Path("/content/gradle-8.13")
repo_url="https://github.com/rodineygarcia/skills-copilot-codespaces-vscode.git"
if not root.exists():
    run(["git","clone","--depth","1","-b","ascii-aventura-android-apk",repo_url,str(root)])

# SDK oficial Google, ferramenta de linha de comando
sdk_tool=sdk/"cmdline-tools"/"latest"/"bin"/"sdkmanager"
if not sdk_tool.exists():
    sdk.mkdir(parents=True,exist_ok=True)
    archive=Path("/content/commandline-tools.zip")
    url="https://dl.google.com/android/repository/commandlinetools-linux-15859902_latest.zip"
    print("Baixando Android SDK Command Line Tools...", flush=True)
    urllib.request.urlretrieve(url,archive)
    unpack=Path("/content/android-tools-temp")
    unpack.mkdir(exist_ok=True)
    with zipfile.ZipFile(archive) as z:z.extractall(unpack)
    (sdk/"cmdline-tools").mkdir(parents=True,exist_ok=True)
    shutil.move(str(unpack/"cmdline-tools"),str(sdk/"cmdline-tools"/"latest"))

env=os.environ.copy()
env["ANDROID_HOME"]=str(sdk)
env["ANDROID_SDK_ROOT"]=str(sdk)
env["PATH"]=str(sdk_tool.parent)+":"+env["PATH"]
if Path("/usr/lib/jvm/java-17-openjdk-amd64").exists():
    env["JAVA_HOME"]="/usr/lib/jvm/java-17-openjdk-amd64"
    env["PATH"]=env["JAVA_HOME"]+"/bin:"+env["PATH"]
run([str(sdk_tool),"--sdk_root="+str(sdk),"--licenses"],input=("y\n"*150),text=True,env=env,stdout=subprocess.DEVNULL)
run([str(sdk_tool),"--sdk_root="+str(sdk),"platforms;android-36","build-tools;36.0.0"],env=env)
(root/"local.properties").write_text("sdk.dir="+str(sdk)+"\n")
if not gradle_dir.exists():
    archive=Path("/content/gradle-8.13-bin.zip")
    print("Baixando Gradle 8.13...",flush=True)
    urllib.request.urlretrieve("https://services.gradle.org/distributions/gradle-8.13-bin.zip",archive)
    with zipfile.ZipFile(archive) as z:z.extractall("/content")
gradle=str(gradle_dir/"bin"/"gradle")
print("Compilando o aplicativo...",flush=True)
run([gradle,"--no-daemon",":app:assembleDebug","--stacktrace"],cwd=str(root),env=env)
apk=root/"app"/"build"/"outputs"/"apk"/"debug"/"app-debug.apk"
assert apk.is_file(), "O APK nao foi gerado."
print("APK criado com sucesso:",apk,apk.stat().st_size,"bytes",flush=True)
files.download(str(apk))